### Simulations

In [1]:
import numpy as np
import networkx as nx
import random
import torch
import torch.nn as nn
from torch_geometric.data import Data
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import accuracy_score
import random
from models import GEE, GNN
from time import time
import matplotlib.pyplot as plt
from pathlib import Path  
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

def generate_dc_sbm(n, k, pq, r, degree_dist="beta", params=(1, 4), balanced="True", seed=None):
    """
    Generates a graph according to the Degree-Corrected Stochastic Block Model (DC-SBM).

    In DC-SBM, the probability of an edge between node i and j is:
    P(i, j) = degree_params[i] * degree_params[j] * probs[labels[i], labels[j]]

    Args:
        n (int): Total number of nodes.
        k (int): Number of communities (blocks).
        pq (float): The base probability for intra-community edges (p).
        r (float): The base probability for inter-community edges (q).
        degree_dist (str, optional): The distribution for sampling degree parameters ('beta' or 'uniform').
        params (tuple, optional): The parameters for the chosen degree distribution.
        balanced (bool, optional): If True, communities have nearly equal sizes.
                                   If False, sizes are imbalanced based on a fixed ratio.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        tuple: A tuple containing (networkx.Graph, numpy.ndarray) for the graph and labels.
    """
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    if balanced:
        # Distribute nodes evenly across k communities
        block_sizes = [n // k] * k
        remainder = n - sum(block_sizes)
        for i in range(remainder):
            block_sizes[i] += 1

        labels = np.concatenate([[i] * block_sizes[i] for i in range(k)])
    else:
        ratio = np.array([1, 2, 3, 4], dtype=float)

        block_sizes = np.floor(ratio / ratio.sum() * n).astype(int)

        remainder = n - block_sizes.sum()
        block_sizes[:remainder] += 1         

        labels = np.concatenate([[i] * block_sizes[i] for i in range(k)])

    # Sample degree parameters
    if degree_dist == "beta":
        degree_params = np.random.beta(params[0], params[1], size=n)
    elif degree_dist == "uniform":
        degree_params = np.random.uniform(params[0], params[1], size=n)
    else:
        raise ValueError("Unsupported degree distribution. Use 'beta' or 'uniform'.")
    G = nx.Graph()
    probs = np.full((k, k), r)
    np.fill_diagonal(probs, pq)

    # Add edges based on degree-corrected probabilities
    for i in range(n):
        for j in range(i + 1, n):
            prob = probs[labels[i], labels[j]] * degree_params[i] * degree_params[j]
            if np.random.rand() < prob:
                G.add_edge(i, j)

    return G, labels

def generate_sbm(n, k, pq, r, balanced="True", seed=None):
    """
    Generates a graph according to the standard Stochastic Block Model (SBM).

    In SBM, the probability of an edge between node i and j depends only on their communities.
    P(i, j) = probs[labels[i], labels[j]]

    Args:
        n (int): Total number of nodes.
        k (int): Number of communities (blocks).
        pq (float): The probability for intra-community edges (p).
        r (float): The probability for inter-community edges (q).
        balanced (bool, optional): If True, communities have nearly equal sizes.
                                   If False, sizes are imbalanced based on a fixed ratio.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        tuple: A tuple containing (networkx.Graph, numpy.ndarray) for the graph and labels.
    """
    if seed != None:
        np.random.seed(seed)
        random.seed(seed)
    
    if balanced:
        # Distribute nodes evenly across k communities
        block_sizes = [n // k] * k
        remainder = n - sum(block_sizes)
        for i in range(remainder):
            block_sizes[i] += 1

        labels = np.concatenate([[i] * block_sizes[i] for i in range(k)])
    else:
        ratio = np.array([1, 2, 3, 4], dtype=float)

        block_sizes = np.floor(ratio / ratio.sum() * n).astype(int)

        remainder = n - block_sizes.sum()
        block_sizes[:remainder] += 1         

        labels = np.concatenate([[i] * block_sizes[i] for i in range(k)])

    G = nx.Graph()
    probs = np.full((k, k), r) 
    np.fill_diagonal(probs, pq) 

    for i in range(n):
        for j in range(i + 1, n):
            comm_i = labels[i]
            comm_j = labels[j]
            if np.random.rand() < probs[comm_i, comm_j]:
                G.add_edge(i, j)  

    return G, labels

def make_cv_splits(labels, num_folds=2, val_ratio=0.1, n_reps=10, seed=0):
    """
    Generates repeated k-fold cross-validation splits with a dedicated validation set.


    Args:
        labels: A 1D NumPy array or PyTorch tensor of ground-truth labels, used for
                stratification when creating the validation set.
        num_folds (int): The number of folds for the primary K-Fold split (the 'k').
        val_ratio (float): The proportion of the initial training set to be used
                           for validation. For example, 0.1 means 10% of the
                           train_val indices become the validation set.
        n_reps (int): The number of times to repeat the entire k-fold splitting process.
        seed (int): The random seed for reproducibility.

    Returns:
        list: A nested list structured as `list[repetition][fold]`. Each element is
              a dictionary containing 'train', 'val', and 'test' indices as
              PyTorch LongTensors.
    """
    torch.manual_seed(seed)
    y = labels.cpu().numpy() if torch.is_tensor(labels) else labels
    all_splits = []

    for rep in range(n_reps):
        kf = KFold(n_splits=num_folds, shuffle=True,
                              random_state=rep)
        rep_splits = []

        for test_idx, train_val_idx in kf.split(np.arange(len(y)), y):
            train_idx, val_idx = train_test_split(
                train_val_idx,
                test_size=val_ratio,
                random_state=rep,
                stratify=y[train_val_idx]
            )
            rep_splits.append({
                "train": torch.tensor(train_idx, dtype=torch.long),
                "val"  : torch.tensor(val_idx,   dtype=torch.long),
                "test" : torch.tensor(test_idx,  dtype=torch.long)
            })
        all_splits.append(rep_splits)
    return all_splits

def stratified_split(labels, train_ratio=0.18, val_ratio=0.02, test_ratio=0.80, n_repeats=100, seed=0):
    """Creates repeated stratified train/validation/test splits for labels.

    Args:
        labels: A 1D array or tensor of node labels.
        train_ratio (float): The approximate proportion for the training set.
        val_ratio (float): The approximate proportion for the validation set.
        n_repeats (int): The number of distinct splits to generate.
        seed (int): The random seed for reproducibility.

    Returns:
        list: A list of dictionaries, each holding 'train', 'val', 'test' indices.
    """
    y = labels.cpu().numpy() if torch.is_tensor(labels) else labels
    rng = np.random.default_rng(seed)
    num_classes = np.unique(y).size
    all_splits = []
    for repeat in range(n_repeats):
        train_idx, val_idx, test_idx = [], [], []
        for c in np.unique(y):

            idx = np.where(y == c)[0]
            idx = rng.permutation(idx)
            n = len(idx)
            if n == 3:
                train_idx.append(idx[:2])
                val_idx.append(idx[2])
            if n <= 2:
                raise ValueError(f"Not enough samples")
            else:
                n_train = max(2, int(np.round(train_ratio * n)))
                n_val = max(1, int(np.round(val_ratio * n)))
                n_test = n - n_train - n_val

                train_idx.append(idx[:n_train])
                val_idx.append(idx[n_train:n_train+n_val])
                test_idx.append(idx[n_train+n_val:])

        train_idx = np.concatenate(train_idx)
        val_idx = np.concatenate(val_idx)
        test_idx = np.concatenate(test_idx)
        all_splits.append({
            'train': train_idx,
            'val': val_idx,
            'test': test_idx,
        })
    return all_splits

def write(file_path, content, r):
    """
    Appends experiment results for a given run to a tab-separated file.

    Args:
        file_path (str): The path to the output file.
        content (dict): A dictionary mapping method names to their results.
        r (int): The inter-community probability.
    """
    with open(file_path, "a") as f:
        for method, arr in content.items():
            arr = np.asarray(arr, dtype=float)
            flat = arr.flatten()
            ari_str = ",".join(f"{x}" for x in flat)
            line = f"{r}\t{method}\t{ari_str}\n"
            f.write(line)

def lda_eval(emb, idx, y_true, solver="svd", shrinkage=None):
    """
    Evaluates node embeddings using a Linear Discriminant Analysis (LDA) classifier.

    Args:
        emb: A 2D array or tensor of node embeddings.
        idx (dict): A dictionary containing 'train', 'val', and 'test' indices.
        y_true (torch.Tensor): The ground-truth labels for all nodes.

    Returns:
        float: The classification accuracy on the test set.
    """
    if isinstance(emb, torch.Tensor):
        emb = emb.detach().cpu().numpy()

    idx_train_all = np.concatenate([idx["train"], idx["val"]])
    idx_test = idx["test"]

    X_train, y_train = emb[idx_train_all], y_true[idx_train_all]
    X_test,  y_test  = emb[idx_test],      y_true[idx_test]

    lda = LinearDiscriminantAnalysis(solver=solver, shrinkage=shrinkage)
    lda.fit(X_train, y_train)
    y_pred = lda.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    return acc


In [2]:
def run_simulation(data, edge_list, cv_splits, *, gnn_kwargs = None):
    """
    Orchestrates a comparative evaluation of multiple models over k-fold cross-validation.

    Args:
        data: The PyTorch Geometric `Data` object for the dataset.
        edge_list (list): The edge list format required by the GEE model.
        cv_splits (list): A nested list of splits, structured as
                          `list[repetition][fold]`, where each element is a
                          dictionary of train/val/test indices.
        gnn_kwargs (dict, optional): Keyword arguments to be passed to the
                                     GNN training function.

    Returns:
        tuple: A tuple containing:
               - `acc_all` (dict): A dictionary where keys are model names and
                 values are lists of lists, storing fold accuracies for each repetition.
               - `time_all` (dict): A similar dictionary storing execution times.
               - `grad_infs` (tuple or None): A tuple with diagnostic information
                 (gradient norms, losses) from the first run, or None if not collected.
    """
    torch.manual_seed(0)
    acc_all = {'GEE':[], 'GNN':[], "GG":[], "GG2":[]}
    time_all = {'GEE':[], 'GNN':[], "GG":[], "GG2":[]}
    loss_all = {}
    grad_all = {}

    n_reps = len(cv_splits)
    num_folds = len(cv_splits[0])

    for rep in range(n_reps):
        acc_lists = {'GEE':[], 'GNN':[], "GG":[], "GG2":[]}
        time_lists = {'GEE':[], 'GNN':[], "GG":[], "GG2":[]}
        stats_acc = {}
        stats_time = {}
        print(f"\n=== Run {rep+1}/{n_reps} ===")
        for fold in range(num_folds):
            idx = cv_splits[rep][fold]
            y_true = data.y[idx["test"]]

            # ----------  GEE ----------
            y_train = np.asarray(data.y).copy().reshape(-1,1)
            y_train[idx["val"]] = -1
            y_train[idx["test"]] = -1
            start=time(); Z = GEE.GEE(data.num_nodes, edge_list, y_train); end=time(); t_GEE= end-start
            # y_pred_GEE = torch.argmax(Z, dim=1)[idx["test"]]
            # acc_GEE = accuracy_score(y_true, y_pred_GEE)

            
            # ----------  GNN, GG ----------    
            d_GNN = data.clone()
            d_GG = data.clone()
            d_GG.x = Z
            d_GG2 = data.clone()
            d_GG2.x = Z
            for m in ("train", "val", "test"):
                mask = torch.zeros(d_GNN.num_nodes, dtype=torch.bool)
                mask[idx[m]] = True
                setattr(d_GNN, f"{m}_mask", mask)
                setattr(d_GG, f"{m}_mask", mask)
                setattr(d_GG2, f"{m}_mask", mask)
            
            if rep == 0 and fold == 0:
                gnn_kwargs['return_grad'] = True

            start=time(); model_GNN, logits_GNN, grad_inf_GNN = GNN.train(d_GNN, **gnn_kwargs); end=time(); t_GNN= end-start
            y_pred_GNN = logits_GNN.argmax(dim=1)[idx["test"]]
            acc_GNN = accuracy_score(y_true, y_pred_GNN)

            start=time(); model_GG, logits_GG, grad_inf_GG = GNN.train(d_GG, **gnn_kwargs); end=time(); t_GG= end-start
            y_pred_GG = logits_GG.argmax(dim=1)[idx["test"]]
            acc_GG = accuracy_score(y_true, y_pred_GG)

            Z_std  = StandardScaler().fit_transform(Z.detach().cpu().numpy())
            GG_std = StandardScaler().fit_transform(logits_GG.detach().cpu().numpy())

            logits_GG2  = np.concatenate([GG_std, Z_std], axis=1)

            acc_GEE = lda_eval(Z,          idx, data.y)
            # acc_GNN = lda_eval(logits_GNN, idx, data.y)
            # acc_GG  = lda_eval(logits_GG,  idx, data.y)
            acc_GG2 = lda_eval(logits_GG2, idx, data.y)

            if gnn_kwargs['return_grad'] == True:
                loss_all['GNN'] = grad_inf_GNN[0]
                loss_all['GG'] = grad_inf_GG[0]
                grad_all['GNN'] = grad_inf_GNN[1]
                grad_all['GG'] = grad_inf_GG[1]
                gnn_kwargs['return_grad'] = False

            acc_list = [acc_GEE, acc_GNN, acc_GG, acc_GG2]
            time_list = [t_GEE, t_GNN, t_GG, t_GG]
            for i, m in enumerate(acc_lists):
                acc_lists[m].append(acc_list[i])
                time_lists[m].append(time_list[i])

            # print(f"  Fold {fold+1}: GEE = {acc_GEE:.4f}, GNN = {acc_GNN:.4f}, GG = {acc_GG:.4f}")

        for m in acc_lists:
            arr = np.array(acc_lists[m])
            stats_acc[m] = arr.mean(), arr.std(ddof=1)/np.sqrt(5)

            # arr = np.array(time_lists[m])
            # stats_time[m] = arr.mean(), arr.std(ddof=1)/np.sqrt(5)

        print(f"\nRun {rep} (5-fold),  GEE = {stats_acc['GEE'][0]:.4f}±{stats_acc['GEE'][1]:.4f}, \
            GNN = {stats_acc['GNN'][0]:.4f}±{stats_acc['GNN'][1]:.4f},\
                GG = {stats_acc['GG'][0]:.4f}±{stats_acc['GG'][1]:.4f},\
                    GG2 = {stats_acc['GG2'][0]:.4f}±{stats_acc['GG2'][1]:.4f}")
        for m in acc_all:
            acc_all[m].append(acc_lists[m])
            time_all[m].append(time_lists[m])
    
    grad_infs = grad_all, loss_all if grad_inf_GNN != None else None
    
    return acc_all, time_all, grad_infs

def run_realdata(data, edge_list, cv_splits, *, gnn_kwargs = None):
    """Orchestrates a comparative experiment of different models on a dataset.

    Args:
        data: The PyTorch Geometric `Data` object for the dataset.
        edge_list (list): The edge list format required by the GEE model.
        cv_splits (list): A list of train/val/test splits, one for each repetition.
        gnn_kwargs (dict, optional): Keyword arguments for the GNN training function.

    Returns:
        tuple: A tuple containing dictionaries for accuracies (`acc_all`),
               execution times (`time_all`), and diagnostic info (`grad_infs`).
    """
    torch.manual_seed(0)
    acc_all = {'GEE':[], 'GNN':[], "GG":[], "GG2":[]}
    time_all = {'GEE':[], 'GNN':[], "GG":[], "GG2":[]}
    loss_all = {}
    grad_all = {}
    stats_acc = {}
    stats_time = {}
    n_reps = len(cv_splits)

    for rep in range(n_reps):
        idx = cv_splits[rep]
        y_true = data.y[idx["test"]]

        # ----------  GEE ----------
        y_train = np.asarray(data.y).copy().reshape(-1,1)
        y_train[idx["val"]] = -1
        y_train[idx["test"]] = -1
        start=time(); Z = GEE.GEE(data.num_nodes, edge_list, y_train); end=time(); t_GEE= end-start
        # y_pred_GEE = torch.argmax(Z, dim=1)[idx["test"]]
        # acc_GEE = accuracy_score(y_true, y_pred_GEE)

        
        # ----------  GNN, GG ----------    
        d_GNN = data.clone()
        d_GG = data.clone()
        d_GG.x = Z
        d_GG2 = data.clone()
        d_GG2.x = Z
        for m in ("train", "val", "test"):
            mask = torch.zeros(d_GNN.num_nodes, dtype=torch.bool)
            mask[idx[m]] = True
            setattr(d_GNN, f"{m}_mask", mask)
            setattr(d_GG, f"{m}_mask", mask)
            setattr(d_GG2, f"{m}_mask", mask)
        
        if rep == 0:
            gnn_kwargs['return_grad'] = True

        start=time(); model_GNN, logits_GNN, grad_inf_GNN = GNN.train(d_GNN, **gnn_kwargs); end=time(); t_GNN= end-start
        y_pred_GNN = logits_GNN.argmax(dim=1)[idx["test"]]
        acc_GNN = accuracy_score(y_true, y_pred_GNN)

        start=time(); model_GG, logits_GG, grad_inf_GG = GNN.train(d_GG, **gnn_kwargs); end=time(); t_GG= end-start
        y_pred_GG = logits_GG.argmax(dim=1)[idx["test"]]
        acc_GG = accuracy_score(y_true, y_pred_GG)

        Z_std  = StandardScaler().fit_transform(Z.detach().cpu().numpy())
        GG_std = StandardScaler().fit_transform(logits_GG.detach().cpu().numpy())

        logits_GG2  = np.concatenate([GG_std, Z_std], axis=1)
        
        acc_GEE = lda_eval(Z,          idx, data.y)
        # acc_GNN = lda_eval(logits_GNN, idx, data.y)
        # acc_GG  = lda_eval(logits_GG,  idx, data.y)
        acc_GG2 = lda_eval(logits_GG2, idx, data.y)

        if gnn_kwargs['return_grad'] == True:
            loss_all['GNN'] = grad_inf_GNN[0]
            loss_all['GG'] = grad_inf_GG[0]
            grad_all['GNN'] = grad_inf_GNN[1]
            grad_all['GG'] = grad_inf_GG[1]
            gnn_kwargs['return_grad'] = False

        acc_list = [acc_GEE, acc_GNN, acc_GG, acc_GG2]
        time_list = [t_GEE, t_GNN, t_GG, t_GG]
        for i, m in enumerate(acc_all):
            acc_all[m].append(acc_list[i])
            time_all[m].append(time_list[i])

            # print(f"  Fold {fold+1}: GEE = {acc_GEE:.4f}, GNN = {acc_GNN:.4f}, GG = {acc_GG:.4f}")

    for m in acc_all:
        arr = np.array(acc_all[m])
        stats_acc[m] = arr.mean(), arr.std(ddof=1)/np.sqrt(100)

        # arr = np.array(time_lists[m])
        # stats_time[m] = arr.mean(), arr.std(ddof=1)/np.sqrt(5)

    print(f"GEE = {stats_acc['GEE'][0]:.4f}±{stats_acc['GEE'][1]:.4f}, \
        GNN = {stats_acc['GNN'][0]:.4f}±{stats_acc['GNN'][1]:.4f},\
            GG = {stats_acc['GG'][0]:.4f}±{stats_acc['GG'][1]:.4f},\
                GG2 = {stats_acc['GG2'][0]:.4f}±{stats_acc['GG2'][1]:.4f}")


    grad_infs = grad_all, loss_all if grad_inf_GNN != None else None
    
    return acc_all, time_all, grad_infs

In [ ]:
# =============================================================================
# Main script for the simulation study on SBM and DC-SBM graphs.
#
# This script orchestrates a comprehensive simulation to evaluate and compare
# model performance under varying graph structures. It systematically iterates
# through a range of inter-community edge probabilities ('r') for either a
# standard SBM or a Degree-Corrected SBM (controlled by 'flag').
#
# The evaluation is conducted across multiple k-fold cross-validation setups
# (e.g., k=2, 5, 10, 20), and for each parameter combination (r, k), multiple
# independent replications are run to ensure robust statistical assessment.
# =============================================================================
torch.manual_seed(901)
random.seed(901)
np.random.seed(901)
flag = "DC_SBM" # "SBM"
balanced = False
bal = "" if balanced else "unbal" # ""
fold_list = [2, 5, 10, 20] if flag == "DC_SBM" else [2, 5, 10]
for num_folds in fold_list:
    percent = int(100 / num_folds)
    n_reps = int( 200 / num_folds)
    n, k, pq = 2000, 4, 0.3
    rvalues = np.arange(0, 0.305, 0.02)

    # n, k, pq = 800, 4, 0.15
    # rvalues = np.arange(0, 0.151, 0.01)

    for r in rvalues:
        G, Y = generate_dc_sbm(n, k, pq, r, balanced=balanced)
        G.add_edges_from((i, i) for i in range(n))

        edge_list = [(u, v, 1) for u, v in G.edges()]
        labels = torch.tensor(Y, dtype=torch.long)
        edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()
        cv_splits = make_cv_splits(Y, num_folds=num_folds, val_ratio=0.1,
                                    n_reps=n_reps)
        gnn_kwargs = dict(k=k, lr=0.001, num_epochs=10000, patience=100, concat= False, return_grad=False)

        initializer = nn.init.xavier_uniform_
        z = initializer(torch.empty(n, k))
        data = Data(x=z, edge_index=edge_index, y=labels, k=k)
        acc_all, time_all, grad_infs = run_simulation(data, edge_list, cv_splits, gnn_kwargs=gnn_kwargs)

        write(f"results/{flag}/{percent}%/acc_{bal}.txt", acc_all, r)
        write(f"results/{flag}/{percent}%/time_{bal}.txt", time_all, r)


In [ ]:
# =============================================================================
# Main script for the simulation study on SBM and DC-SBM graphs.
#
# NOTE: This script is designed to handle class imbalance, a common issue
#       in real-world graphs or simulations with small training data splits.
#       It uses a custom `stratified_split` function to ensure that every
#       class has a sufficient number of nodes in the training set, which is
#       crucial for stable model training and evaluation.
# =============================================================================
torch.manual_seed(901)
random.seed(901)
np.random.seed(901)

flag = "SBM" 
balanced = False
bal = "" if balanced else "unbal" # ""
n_reps = 200
percent = 5 

n, k, pq = 800, 4, 0.15
rvalues = np.arange(0, 0.151, 0.01)

for r in rvalues:
    G, Y = generate_sbm(n, k, pq, r, balanced=balanced)
    G.add_edges_from((i, i) for i in range(n))

    edge_list = [(u, v, 1) for u, v in G.edges()]
    labels = torch.tensor(Y, dtype=torch.long)
    edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()

    cv_splits = stratified_split(Y, train_ratio=0.9/num_folds, val_ratio=0.1/num_folds, test_ratio=1/num_folds, n_repeats=n_reps)

    gnn_kwargs = dict(k=k, lr=0.001, num_epochs=10000, patience=100, concat= False, return_grad=False)

    initializer = nn.init.xavier_uniform_
    z = initializer(torch.empty(n, k))
    data = Data(x=z, edge_index=edge_index, y=labels, k=k)
    acc_all, time_all, grad_infs = run_realdata(data, edge_list, cv_splits, gnn_kwargs=gnn_kwargs)

    write(f"results/{flag}/{percent}%/acc_{bal}.txt", acc_all, r)
    write(f"results/{flag}/{percent}%/time_{bal}.txt", time_all, r)




